In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [ ]:
import numpy as np
import plotly.graph_objects as go
import models.black_scholes as bs

DS = 50      
DJ = 50      
r = 0.01     
sigma = 0.25 
T = 1.0      

### Opgave 1
Værdien af egenkapitalen og obligationerne som funktion af $V$.

In [3]:
V = np.linspace(0.01, 200, 500)

equity = np.array([bs.call(v, DS + DJ, sigma, r, T) for v in V])
senior_debt = np.array([v - bs.call(v, DS, sigma, r, T) for v in V])
junior_debt = np.array([bs.call(v, DS, sigma, r, T) - bs.call(v, DS + DJ, sigma, r, T) for v in V])

fig = go.Figure()
fig.add_trace(go.Scatter(x=V, y=equity, mode="lines", name="Equity"))
fig.add_trace(go.Scatter(x=V, y=senior_debt, mode="lines", name="Senior debt"))
fig.add_trace(go.Scatter(x=V, y=junior_debt, mode="lines", name="Junior debt"))

fig.update_layout(
    title="Equity and bond values as a function of V",
    xaxis_title="V (firm value)",
    yaxis_title="Value",
    template="plotly_white",
)
fig.show()

### Opgave 2
Kreditspænd for senior gæld og junior gæld som funktion af tid til udløb $T$. Inkluder tilfældene $V = 90$, $V = 130$ (grunddata) og $V = 180$.

In [4]:
def senior_debt_value(V, DS, DJ, sigma, r, T):
    return V - bs.call(V, DS, sigma, r, T)

def junior_debt_value(V, DS, DJ, sigma, r, T):
    return bs.call(V, DS, sigma, r, T) - bs.call(V, DS + DJ, sigma, r, T)

def credit_spread_senior(V, DS, DJ, sigma, r, T):
    if T <= 0:
        return np.nan
    B = senior_debt_value(V, DS, DJ, sigma, r, T)
    if B <= 0:
        return np.inf
    return max(0.0, -np.log(B / DS) / T - r)

def credit_spread_junior(V, DS, DJ, sigma, r, T):
    if T <= 0:
        return np.nan
    B = junior_debt_value(V, DS, DJ, sigma, r, T)
    if B <= 0:
        return np.inf
    return max(0.0, -np.log(B / DJ) / T - r)

T_grid = np.linspace(0.1, 3, 200)
V_values = [90, 130, 180]

fig = go.Figure()
for V in V_values:
    s_senior = [credit_spread_senior(V, DS, DJ, sigma, r, t) for t in T_grid]
    fig.add_trace(go.Scatter(x=T_grid, y=s_senior, mode="lines", name=f"Senior, V={V}"))
fig.update_layout(
    title="Kreditspænd senior gæld som funktion af tid til udløb T",
    xaxis_title="T (år)",
    yaxis_title="Kreditspænd",
    template="plotly_white",
)
fig.show()

fig2 = go.Figure()
for V in V_values:
    s_junior = [credit_spread_junior(V, DS, DJ, sigma, r, t) for t in T_grid]
    fig2.add_trace(go.Scatter(x=T_grid, y=s_junior, mode="lines", name=f"Junior, V={V}"))
fig2.update_layout(
    title="Kreditspænd junior gæld som funktion af tid til udløb T",
    xaxis_title="T (år)",
    yaxis_title="Kreditspænd",
    template="plotly_white",
)
fig2.show()

---

## Springdiffusions-modellen

Værdien af virksomhedens aktiver følger en springdiffusion med lognormalfordelte spring:  
$dV_t = (r - \lambda k) V_t \, dt + \sigma V_t \, dW_t + (Y-1) V_{t-} \, dN_t$,  
hvor $N$ er Poisson($\lambda$) og $\log Y \sim N(\gamma, \delta^2)$. Der gælder $k = \mathbb{E}[Y-1] = e^{\gamma + \delta^2/2} - 1$.

**Grunddata (som før + springparametre):** $V=130$, $D_S=50$, $D_J=50$, $r=0{,}01$, $\sigma=0{,}25$, $T=1$, samt $\lambda=0{,}2$, $\gamma=\ln(0{,}8)$ (svarende til $k=-0{,}2$), $\delta=0{,}4$.

Priser på egenkapital og gæld beregnes ved Monte Carlo: vi simulerer $V_T$ under springdiffusionen og diskonterer udbetalinger ved udløb.

In [5]:
# Springdiffusion: parametre
lam = 0.2
gamma = np.log(0.8)   # k ≈ -0.2
delta = 0.4
k = np.exp(gamma + 0.5 * delta**2) - 1   # E[Y-1]

def simulate_VT_jump_diffusion(V0, r, sigma, lam, gamma, delta, T, n_paths, seed=None):
    """Simulerer V_T under springdiffusion; returnerer array af størrelse n_paths."""
    rng = np.random.default_rng(seed)
    N = rng.poisson(lam * T, size=n_paths)
    Z1 = rng.standard_normal(n_paths)
    Z2 = rng.standard_normal(n_paths)
    log_VT = np.log(V0) + (r - lam * k - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z1
    log_VT += np.where(N > 0, N * gamma + np.sqrt(N) * delta * Z2, 0.0)
    return np.exp(log_VT)

def price_equity_jump(V0, DS, DJ, r, sigma, lam, gamma, delta, T, n_paths=100_000, seed=None):
    VT = simulate_VT_jump_diffusion(V0, r, sigma, lam, gamma, delta, T, n_paths, seed)
    payoff = np.maximum(VT - (DS + DJ), 0.0)
    return np.exp(-r * T) * np.mean(payoff)

def price_senior_jump(V0, DS, DJ, r, sigma, lam, gamma, delta, T, n_paths=100_000, seed=None):
    VT = simulate_VT_jump_diffusion(V0, r, sigma, lam, gamma, delta, T, n_paths, seed)
    payoff = np.minimum(VT, DS)
    return np.exp(-r * T) * np.mean(payoff)

def price_junior_jump(V0, DS, DJ, r, sigma, lam, gamma, delta, T, n_paths=100_000, seed=None):
    VT = simulate_VT_jump_diffusion(V0, r, sigma, lam, gamma, delta, T, n_paths, seed)
    payoff = np.minimum(np.maximum(VT - DS, 0.0), DJ)
    return np.exp(-r * T) * np.mean(payoff)

# Eksempel: priser ved grunddata
n_paths = 100_000
S_jd = price_equity_jump(130, DS, DJ, r, sigma, lam, gamma, delta, T, n_paths, seed=42)
B_senior_jd = price_senior_jump(130, DS, DJ, r, sigma, lam, gamma, delta, T, n_paths, seed=42)
B_junior_jd = price_junior_jump(130, DS, DJ, r, sigma, lam, gamma, delta, T, n_paths, seed=42)
print("Springdiffusion (grunddata): Equity =", round(S_jd, 4), ", Senior =", round(B_senior_jd, 4), ", Junior =", round(B_junior_jd, 4))

Springdiffusion (grunddata): Equity = 34.932 , Senior = 49.3749 , Junior = 45.8476


### Opgave 3
Gentag spørgsmål 2 i springdiffusions-modellen: kreditspænd for senior og junior gæld som funktion af tid til udløb for $V = 90$, $V = 130$ og $V = 180$.

In [8]:
def credit_spread_senior_jump(V, DS, DJ, r, sigma, lam, gamma, delta, T, n_paths=50_000, seed=None):
    if T <= 0:
        return np.nan
    B = price_senior_jump(V, DS, DJ, r, sigma, lam, gamma, delta, T, n_paths, seed)
    if B <= 0:
        return np.inf
    return max(0.0, -np.log(B / DS) / T - r)

def credit_spread_junior_jump(V, DS, DJ, r, sigma, lam, gamma, delta, T, n_paths=50_000, seed=None):
    if T <= 0:
        return np.nan
    B = price_junior_jump(V, DS, DJ, r, sigma, lam, gamma, delta, T, n_paths, seed)
    if B <= 0:
        return np.inf
    return max(0.0, -np.log(B / DJ) / T - r)

T_grid_jd = np.linspace(0.2, 3, 15)   # færre punkter pga. MC
V_values = [90, 130, 180]
n_paths_mc = 50_000

fig_s = go.Figure()
for V in V_values:
    s_sen = [credit_spread_senior_jump(V, DS, DJ, r, sigma, lam, gamma, delta, t, n_paths_mc, seed=42 + int(V)) for t in T_grid_jd]
    fig_s.add_trace(go.Scatter(x=T_grid_jd, y=s_sen, mode="lines+markers", name=f"Senior, V={V}"))
fig_s.update_layout(
    title="Springdiffusion: kreditspænd senior gæld vs. tid til udløb T",
    xaxis_title="T (år)",
    yaxis_title="Kreditspænd",
    template="plotly_white",
)
fig_s.show()

fig_j = go.Figure()
for V in V_values:
    s_jun = [credit_spread_junior_jump(V, DS, DJ, r, sigma, lam, gamma, delta, t, n_paths_mc, seed=42 + int(V)) for t in T_grid_jd]
    fig_j.add_trace(go.Scatter(x=T_grid_jd, y=s_jun, mode="lines+markers", name=f"Junior, V={V}"))
fig_j.update_layout(
    title="Springdiffusion: kreditspænd junior gæld vs. tid til udløb T",
    xaxis_title="T (år)",
    yaxis_title="Kreditspænd",
    template="plotly_white",
)
fig_j.show()

### Eksperiment: kilden til volatilitet 

Vi holder den **samlede kvadratvariation** konstant (QV = σ² + λ·E[(log Y)²] = σ² + λ(γ²+δ²)) og skifter mellem mere volatilitet fra **diffusion** (høj σ, lav λ) og mere fra **spring** (lav σ, høj λ). Graferne viser kreditspænd (total gæld) som funktion af tid til udløb for V = 130, 150, 200.

In [10]:
from plotly.subplots import make_subplots

# Samme γ, δ i begge scenarier; QV = σ² + λ(γ²+δ²) holdes konstant
gamma_fig = np.log(0.8)
delta_fig = 0.4
var_jump = gamma_fig**2 + delta_fig**2   # E[(log Y)²]
QV_target = 0.4

# (a) Lav diffusion-volatilitet → mere fra spring
sigma_a = 0.1
lam_a = (QV_target - sigma_a**2) / var_jump

# (b) Høj diffusion-volatilitet → mindre fra spring
sigma_b = 0.3
lam_b = (QV_target - sigma_b**2) / var_jump

k_fig = np.exp(gamma_fig + 0.5 * delta_fig**2) - 1

def spread_total_debt_jump(V, DS, DJ, r, sigma, lam, gamma, delta, T, n_paths=40_000, seed=None):
    """Kreditspænd for total gæld (senior + junior), face = DS+DJ."""
    if T <= 0:
        return np.nan
    B_s = price_senior_jump(V, DS, DJ, r, sigma, lam, gamma, delta, T, n_paths, seed)
    B_j = price_junior_jump(V, DS, DJ, r, sigma, lam, gamma, delta, T, n_paths, (seed + 1000) if seed is not None else None)
    B = B_s + B_j
    D = DS + DJ
    if B <= 0:
        return np.inf
    return max(0.0, -np.log(B / D) / T - r)

T_fig = np.linspace(0.25, 15, 20)
V_fig = [130, 150, 200]
n_mc = 40_000

def compute_curves(sigma_val, lam_val, seed_base):
    curves = {}
    for V in V_fig:
        curves[V] = [spread_total_debt_jump(V, DS, DJ, r, sigma_val, lam_val, gamma_fig, delta_fig, t, n_mc, seed=seed_base + V + int(t*10)) for t in T_fig]
    return curves

curves_a = compute_curves(sigma_a, lam_a, 100)
curves_b = compute_curves(sigma_b, lam_b, 500)

# Samme farve per V i begge paneler
colors_V = {130: "#1f77b4", 150: "#ff7f0e", 200: "#2ca02c"}

fig_exp = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"(a) σ={sigma_a}, λ≈{lam_a:.2f} — mere fra spring",
        f"(b) σ={sigma_b}, λ≈{lam_b:.2f} — mere fra diffusion",
    ),
    horizontal_spacing=0.12,
)

for V in V_fig:
    fig_exp.add_trace(
        go.Scatter(
            x=T_fig, y=curves_a[V], mode="lines", name=f"V={V}",
            line=dict(width=2, color=colors_V[V]), legendgroup=f"V{V}",
        ),
        row=1, col=1,
    )
for V in V_fig:
    fig_exp.add_trace(
        go.Scatter(
            x=T_fig, y=curves_b[V], mode="lines", name=f"V={V}",
            line=dict(width=2, color=colors_V[V]), legendgroup=f"V{V}", showlegend=False,
        ),
        row=1, col=2,
    )

fig_exp.update_xaxes(title_text="Tid til udløb T (år)", row=1, col=1)
fig_exp.update_xaxes(title_text="Tid til udløb T (år)", row=1, col=2)
fig_exp.update_yaxes(title_text="Kreditspænd", row=1, col=1)
fig_exp.update_yaxes(title_text="Kreditspænd", row=1, col=2)
fig_exp.update_layout(
    title=dict(text="Effekt af kilden til volatilitet (konstant QV): kreditspænd vs. T", font=dict(size=14)),
    template="plotly_white",
    height=460,
    margin=dict(t=100, b=85, l=55, r=40),
    showlegend=True,
    legend=dict(orientation="h", yanchor="top", y=-0.14, xanchor="center", x=0.5, font=dict(size=11)),
    font=dict(size=11),
)
fig_exp.update_annotations(font=dict(size=12))
fig_exp.show()